In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf
from path_config import *


### 规整

对于预测后的数据
1. 按年份
2. 按长短链

In [ ]:
import os
import pandas as pd





def merge_year_one(folder_path, str_y='value', max_year=2021):

    def merge_year_one1(folder_path, str_y='value'):
        all_data = pd.DataFrame()
        for filename in os.listdir(folder_path):
            if filename.endswith('.csv'):
                year = filename.split('_')[1].split('.')[0]
                file_path = os.path.join(folder_path, filename)
                df = pd.read_csv(file_path)
                df = df[['lon_grid', 'lat_grid',str_y]]
                df = df.rename(columns={str_y: year})
                if all_data.empty:
                    all_data = df
                else:

                    all_data = pd.merge(all_data, df, on=['lon_grid', 'lat_grid'], how='outer')

            else:
                pass
        return all_data


    years = [str(i) for i in range(2000, max_year)]


    def calculate_slope(row):

        y = row.dropna().values
        x = np.array(years)[:len(y)].astype(int)


        if len(x) < 2:
            return np.nan


        slope, intercept = np.polyfit(x, y, 1)

        return slope
    all_data = merge_year_one1(folder_path, str_y)
    print(all_data.columns)

    all_data['slope'] = all_data[years].apply(calculate_slope, axis=1)
    all_data['value'] = all_data[years].mean(axis=1)
    return all_data




In [44]:
sw_folder_path = path_part3_sw + 'sw_lgbm_output/only_year/'
sw_path_year = path_part3_sw + 'sw_lgbm_output/final_file/'
df_year_data = merge_year_one(sw_folder_path,'value')
df_year_data.to_csv(sw_path_year + 'sw_year.csv',index=False)

df_year_data_lc = merge_year_one(sw_folder_path,'lc_value')
df_year_data_lc.to_csv(sw_path_year + 'sw_lc_year.csv',index=False)

df_year_data_sc = merge_year_one(sw_folder_path,'sc_value')
df_year_data_sc.to_csv(sw_path_year + 'sw_sc_year.csv',index=False)

Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')
Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')
Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')


In [45]:
lr_folder_path = path_part3_lr + 'lr_lgbm_output/only_year/'
lr_path_year = path_part3_lr + 'lr_lgbm_output/final_file/'
df_year_data = merge_year_one(lr_folder_path,'value')
df_year_data.to_csv(lr_path_year + 'lr_year.csv',index=False)

df_year_data_lc = merge_year_one(lr_folder_path,'lc_value')
df_year_data_lc.to_csv(lr_path_year + 'lr_lc_year.csv',index=False)

df_year_data_sc = merge_year_one(lr_folder_path,'sc_value')
df_year_data_sc.to_csv(lr_path_year + 'lr_sc_year.csv',index=False)

Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')
Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')
Index(['lon_grid', 'lat_grid', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020'],
      dtype='object')


一些网格内并不存在地表水，同样也就不可能存在鱼类，因此需要去除数据中的这些网格


In [ ]:
df_water = pd.read_csv(path_file_csv + 'water.csv')

df_water_crd = df_water[['lon','lat']][df_water['water'] == 1]

dic_file_path = {sw_path_year:['sw_year.csv', 'sw_lc_year.csv', 'sw_sc_year.csv'],
                 lr_path_year:['lr_year.csv', 'lr_lc_year.csv', 'lr_sc_year.csv']}
for path, files in dic_file_path.items():
    for file in files:
        df = pd.read_csv(path + file)

        df = df[df[['lon_grid', 'lat_grid']].apply(tuple, axis=1).isin(df_water_crd.apply(tuple, axis=1))]

        df.to_csv(path + file, index=False)


### 视觉化


In [ ]:

import os
import pandas as pd
from netCDF4 import Dataset
import numpy as np

def change_to_nc(path_new_file, data_type, str_colname='value'):
    name_new_file = f'{data_type}_year_{str_colname}.nc'
    df = pd.read_csv(path_new_file + data_type + '_year.csv')
    df = df[['lon_grid', 'lat_grid', str_colname]]
    nc_file = Dataset(path_new_file + name_new_file, 'w', format='NETCDF4')

    lon = np.arange(-180, 180, 1)
    lat = np.arange(-90, 90, 1)

    nc_file.createDimension('lon', len(lon))
    nc_file.createDimension('lat', len(lat))


    lon_var = nc_file.createVariable('lon', 'f4', 'lon')
    lat_var = nc_file.createVariable('lat', 'f4', 'lat')


    lon_var[:] = lon
    lat_var[:] = lat


    value_var = nc_file.createVariable(str_colname, 'f4', ('lat', 'lon'))


    value_data = np.full((len(lat), len(lon)), np.nan)
    for index, row in df.iterrows():
        if row['lon_grid'] != -180 and row['lon_grid'] != 180 and row['lat_grid'] != -90 and row['lat_grid'] != 90:

            value_data[int(row['lat_grid'] + 90), int(row['lon_grid'] + 180)] = row[str_colname]
    value_var[:, :] = value_data

    nc_file.close()
    return print('over')


In [48]:
path_nf = path_part3_sw + 'sw_lgbm_output/final_file/'
# str_colname = 'slope'

for str_colname in ['slope', 'value']:
    data_type1 = 'sw'
    data_type2 = 'sw_lc'
    data_type3 = 'sw_sc'
    change_to_nc(path_nf, data_type1, str_colname)
    change_to_nc(path_nf, data_type2, str_colname)
    change_to_nc(path_nf, data_type3, str_colname)

over
over
over
over
over
over


In [49]:
path_nf = path_part3_lr + 'lr_lgbm_output/final_file/'
for str_colname in ['slope', 'value']:
    data_type1 = 'lr'
    data_type2 = 'lr_lc'
    data_type3 = 'lr_sc'
    change_to_nc(path_nf, data_type1, str_colname)
    change_to_nc(path_nf, data_type2, str_colname)
    change_to_nc(path_nf, data_type3, str_colname)

over
over
over
over
over
over


### 规整2

对于规整后的数据
1. 全球归一
2. 按大洲归一

In [ ]:

import os
import pandas as pd

def merge_year_file(folder_path, output_file_path):




    file_list = [file for file in os.listdir(folder_path) if file.endswith('.csv')]


    combined_data = pd.DataFrame()


    for file in file_list:
        file_path = os.path.join(folder_path, file)
        data = pd.read_csv(file_path)
        data['year'] = file[3:7]
        combined_data = pd.concat([combined_data, data], ignore_index=True)
    print(combined_data.columns)




    df_country_id = pd.read_csv(path_file_csv + 'country_id.csv')
    df_country_id = df_country_id.rename(columns={'lon':'lon_grid', 'lat':'lat_grid'})
    df_country = pd.read_csv(path_file_csv + 'country.csv')
    df_country = df_country[['region','country_id']]



    combined_data = pd.merge(combined_data, df_country_id, on=['lon_grid', 'lat_grid'], how='left')


    combined_data = pd.merge(combined_data, df_country, on='country_id', how='left')
    combined_data.to_csv(output_file_path, index=False)

    return print("CSV文件合并完成，并保存为：", output_file_path)


In [51]:
sw_folder_path = path_part3_sw + 'sw_lgbm_output/only_year'
sw_output_file_path = path_part3_sw + 'sw_lgbm_output/final_file/sw.csv'
merge_year_file(sw_folder_path, sw_output_file_path)

Index(['lon_grid', 'lat_grid', 'PFNA', 'PFHpA', 'PFOA', 'PFOS', 'FOSA', 'PFDA',
       'PFDS', 'PFBS', 'PFBA', 'PFHxA', 'PFHxS', 'PFTeDA', 'PFDoDA', 'PFUnDA',
       'PFTrDA', 'PFPeDA', 'PFHxDA', 'PFODA', 'PFPeS', 'PFHpS', 'PFNS',
       'MeFOSA', 'EtFOSA', 'MeFOSAA', 'EtFOSAA', '6:2 diPAP', '8:2 diPAP',
       'PFPeA', '6:2 FTSA', 'FBSA', 'PFECHS', '6:2 Cl-PFESA', 'EtFOSE',
       'MeFOSE', 'PF5OHxA', 'PF4OPeA', 'HFPO-DA', '8:2 FTCA', '10:2 FTCA',
       '4:2 FTSA', '8:2 FTSA', '6:8 PFPIA', '7:3 FTCA', 'FOSAA',
       '6:2/8:2 diPAP', 'PFPrS', '10:2 FTOH', '6:2 FTOH', '6:2 FTCA',
       '8:2 FTOH', 'MeFBSE', '6:2 monoPAP', '8:2 monoPAP', 'ADONA', 'PFPrA',
       '10:2 FTSA', 'FHxSA', '3:3 FTCA', '5:3 FTCA', '4:2 FTOH', 'value',
       'lc_value', 'sc_value', 'year'],
      dtype='object')
CSV文件合并完成，并保存为： E:wyy/code_project/running_outcome/final_data/SPDB/part3_forecast\sw_forecast\sw_lgbm_output/final_file/sw.csv


In [52]:
lr_folder_path = path_part3_lr + 'lr_lgbm_output/only_year'
lr_output_file_path = path_part3_lr + 'lr_lgbm_output/final_file/lr.csv'
merge_year_file(lr_folder_path, lr_output_file_path)

Index(['lon_grid', 'lat_grid', 'PFNA', 'PFHpA', 'PFOA', 'PFOS', 'FOSA', 'PFDA',
       'PFDS', 'PFBS', 'PFBA', 'PFHxA', 'PFHxS', 'PFTeDA', 'PFDoDA', 'PFUnDA',
       'PFTrDA', 'PFPeDA', 'PFHxDA', 'PFODA', 'PFPeS', 'PFHpS', 'PFNS',
       'MeFOSA', 'EtFOSA', 'MeFOSAA', 'EtFOSAA', '6:2 diPAP', '8:2 diPAP',
       'PFPeA', '6:2 FTSA', 'FBSA', 'PFECHS', '6:2 Cl-PFESA', 'EtFOSE',
       'MeFOSE', 'PF5OHxA', 'PF4OPeA', 'HFPO-DA', '8:2 FTCA', '10:2 FTCA',
       '4:2 FTSA', '8:2 FTSA', '6:8 PFPIA', '7:3 FTCA', 'FOSAA',
       '6:2/8:2 diPAP', 'PFPrS', '10:2 FTOH', '6:2 FTOH', '6:2 FTCA',
       '8:2 FTOH', 'MeFBSE', '6:2 monoPAP', '8:2 monoPAP', 'ADONA', 'PFPrA',
       '10:2 FTSA', 'FHxSA', '3:3 FTCA', '5:3 FTCA', '4:2 FTOH', 'value',
       'lc_value', 'sc_value', 'year'],
      dtype='object')
CSV文件合并完成，并保存为： E:wyy/code_project/running_outcome/final_data/SPDB/part3_forecast\lr_forecast\lr_lgbm_output/final_file/lr.csv


### 规整3

In [ ]:

df_country_id = pd.read_csv(path_file_csv + 'country_id.csv')
df_country = pd.read_csv(path_file_csv + 'country.csv')

def re_merge_data(df, df_country_id, df_country):
    sw_combined_data = df.copy()
    sw_combined_data_lc = sw_combined_data[['lon_grid','lat_grid','lc_value','year']]
    sw_combined_data_lc['type'] = 'LC-PFAS'
    sw_combined_data_lc = sw_combined_data_lc.rename(columns={'lc_value':'value'})
    sw_combined_data_sc = sw_combined_data[['lon_grid','lat_grid','sc_value','year']]
    sw_combined_data_sc['type'] = 'SC-PFAS'
    sw_combined_data_sc = sw_combined_data_sc.rename(columns={'sc_value':'value'})
    df_merge = pd.concat([sw_combined_data_lc, sw_combined_data_sc],axis=0)
    print(df_merge.shape)

    df_country_id = df_country_id.rename(columns={'lon':'lon_grid', 'lat':'lat_grid'})

    df_country = df_country[['region','country_id']]

    combined_data = pd.merge(df_merge, df_country_id, on=['lon_grid', 'lat_grid'], how='left')

    combined_data = pd.merge(combined_data, df_country, on='country_id', how='left')
    print(combined_data.columns)
    return combined_data


In [ ]:
sw_output_file_path = path_part3_sw + 'sw_lgbm_output/final_file/sw.csv'

sw_combined_data = pd.read_csv(sw_output_file_path)

combined_data = re_merge_data(sw_combined_data, df_country_id, df_country)
combined_data.to_csv(path_part3_sw + 'sw_lgbm_output/final_file/sw_merge.csv',index=False)

C:\Users\dell\AppData\Local\Temp\ipykernel_14180\3098141914.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_combined_data_lc['type'] = 'LC-PFAS'
C:\Users\dell\AppData\Local\Temp\ipykernel_14180\3098141914.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_combined_data_sc['type'] = 'SC-PFAS'


(2698962, 5)
Index(['lon_grid', 'lat_grid', 'value', 'year', 'type', 'country_id',
       'region'],
      dtype='object')


In [55]:
lr_output_file_path = path_part3_lr + 'lr_lgbm_output/final_file/lr.csv'
# sw_path_save = path_1_describe_global_map + "fig5_sw_g.png"
lr_combined_data = pd.read_csv(lr_output_file_path)

combined_data = re_merge_data(lr_combined_data, df_country_id, df_country)
combined_data.to_csv(path_part3_lr + 'lr_lgbm_output/final_file/lr_merge.csv',index=False)

C:\Users\dell\AppData\Local\Temp\ipykernel_14180\3098141914.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_combined_data_lc['type'] = 'LC-PFAS'
C:\Users\dell\AppData\Local\Temp\ipykernel_14180\3098141914.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_combined_data_sc['type'] = 'SC-PFAS'


(2698962, 5)
Index(['lon_grid', 'lat_grid', 'value', 'year', 'type', 'country_id',
       'region'],
      dtype='object')


### 不确定性

In [ ]:

import os
import pandas as pd
from netCDF4 import Dataset
import numpy as np

def change_to_nc(path_final, path_data, data_type, str_colname='value'):
    name_new_file = f'{data_type}_year_{str_colname}.nc'
    df = pd.read_csv(path_data + data_type + '.csv')
    df = df[['lon_grid', 'lat_grid', str_colname]]
    nc_file = Dataset(path_final + name_new_file, 'w', format='NETCDF4')

    lon = np.arange(-180, 180, 1)
    lat = np.arange(-90, 90, 1)

    nc_file.createDimension('lon', len(lon))
    nc_file.createDimension('lat', len(lat))


    lon_var = nc_file.createVariable('lon', 'f4', 'lon')
    lat_var = nc_file.createVariable('lat', 'f4', 'lat')


    lon_var[:] = lon
    lat_var[:] = lat


    value_var = nc_file.createVariable(str_colname, 'f4', ('lat', 'lon'))


    value_data = np.full((len(lat), len(lon)), np.nan)
    for index, row in df.iterrows():
        if row['lon_grid'] != -180 and row['lon_grid'] != 180 and row['lat_grid'] != -90 and row['lat_grid'] != 90:

            value_data[int(row['lat_grid'] + 90), int(row['lon_grid'] + 180)] = row[str_colname]
    value_var[:, :] = value_data

    nc_file.close()
    return print('over')


In [ ]:
df_water = pd.read_csv(path_file_csv + 'water.csv')

df_water_crd = df_water[['lon','lat']][df_water['water'] == 1]


path_final_sw = path_part3_sw + 'sw_lgbm_output/final_file/'
path_data_sw = path_part3_sw + 'sw_lgbm_output/only_pfas/'
path_final_lr = path_part3_lr + 'lr_lgbm_output/final_file/'
path_data_lr = path_part3_lr + 'lr_lgbm_output/only_pfas/'



dic_file_path = {path_data_sw:['sw_value.csv'],
                 path_data_lr:['lr_value.csv']}
for path, files in dic_file_path.items():
    for file in files:
        df = pd.read_csv(path + file)

        df = df[df[['lon_grid', 'lat_grid']].apply(tuple, axis=1).isin(df_water_crd.apply(tuple, axis=1))]

        df.to_csv(path + file, index=False)

In [4]:
change_to_nc(path_final_sw, path_data_sw, 'sw_value', 'value_cv')
change_to_nc(path_final_lr, path_data_lr, 'lr_value', 'value_cv')

over
over
